# Anatomy of a Redemption
### Spark-Based Multi-Source Sentiment Analysis of Cyberpunk 2077

Lab Group 1 — Data Processing 2, SS 2026

Batch pipeline: Reddit comments + Steam reviews over the 34-month window
(Dec 2020 – Dec 2023), sentiment via VADER, aligned against a patch timeline.
**Core thesis:** Reddit recovers faster than Steam.

**Stages:** 0 Config · 1 Ingest+Filter · 2 Cleaning · 3 VADER · 4 Aggregation · 5 Analysis/Plots

> Final, aggregated results go to `results/` (small, in the repo). The report is written from
> these files, not by re-running the pipeline.

## 0 · Configuration & Parameters (central)

All settings in **one block** — prerequisite for reproducibility. `DATA_MODE`
is the only switch between sample (development) and full run (`results/` snapshot). `SEED`
covers everything stochastic (especially LDA).

In [ ]:
import findspark
findspark.init()

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===== Central parameters (one block — reproducibility) =====
DATA_MODE = "sample"            # "sample" | "full"
SEED = 42                       # for everything stochastic (esp. LDA setSeed)
assert DATA_MODE in ("sample", "full")

WINDOW_START = "2020-12-01"     # 34-month window, inclusive bounds
WINDOW_END   = "2023-12-31"
SUBREDDITS   = ["cyberpunkgame", "LowSodiumCyberpunk", "Games", "pcgaming"]
STEAM_APP_ID = 1091500          # Cyberpunk 2077

ANOMALY_Z_THRESHOLD = 3.0       # volume anomaly / review-bombing flag
LDA_NUM_TOPICS = 8              # topic analysis

# ===== Resolve project root robustly (cross-platform: macOS + Windows) =====
# Notebooks have no __file__; the Jupyter CWD is sometimes the project root and
# sometimes notebooks/. So we walk up from the CWD to find the project root.
def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "notebooks").is_dir() and (p / "README.md").exists():
            return p
    return start

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())

# ===== Paths (POSIX strings -> accepted by Spark on macOS AND Windows) =====
_SUB           = "sample" if DATA_MODE == "sample" else "raw"
RAW_REDDIT     = (PROJECT_ROOT / "data" / _SUB / "reddit").as_posix() + "/"
RAW_STEAM      = (PROJECT_ROOT / "data" / _SUB / "steam").as_posix() + "/"
PROCESSED      = (PROJECT_ROOT / "data" / "processed").as_posix() + "/"
RESULTS        = (PROJECT_ROOT / "results").as_posix() + "/"
FIGURES        = (PROJECT_ROOT / "results" / "figures").as_posix() + "/"
# Patch timeline: small + hand-maintained -> lives in the repo (data/sample/), for sample AND full
PATCH_TIMELINE = (PROJECT_ROOT / "data" / "sample" / "patch_timeline.csv").as_posix()
Path(RESULTS).mkdir(exist_ok=True)
Path(FIGURES).mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_MODE={DATA_MODE}  window={WINDOW_START}..{WINDOW_END}  seed={SEED}")

### SparkSession
Local, `local[*]`. Driver RAM increased for the `full` run.

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CyberpunkSentiment")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "200")          # increase for full run if needed
    .config("spark.driver.memory", "8g" if DATA_MODE == "sample" else "12g")
    .config("spark.sql.files.maxPartitionBytes", "128m")    # controls partition size
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 1 · Ingest + Immediate Filter

**Hard rule:** filter immediately on read (subreddits + time window), before anything else.
GB/month should become a fraction early. Spark reads `.zst` dumps directly.

Three inputs, unified to a common schema `source, month, date, text`:
- **Reddit comments** (`body`) + **submissions** (`title` + `selftext`) — both included.
- **Steam reviews** — verified schema (`Review`, `Date Posted` as `M/d/yyyy`).
- Patch timeline (below) is a small side input, not part of the document corpus.

> Currently only **r/cyberpunkgame** is downloaded; the design already scales to all 4 subreddits
> (just drop their dumps into the same folder). `created_utc` may be a string in these dumps, so it
> is cast to `long`. Run note: the sample dirs are empty — for the first test set `DATA_MODE = "full"`.

In [ ]:
# -------------------------------------------------------------
# Reddit: comments (body) + submissions (title + selftext)
# .zst line-delimited JSON; pathGlobFilter skips .DS_Store etc.
# created_utc may be a string in these dumps -> cast to long.
# -------------------------------------------------------------
START = F.unix_timestamp(F.lit(WINDOW_START), "yyyy-MM-dd")
END   = F.unix_timestamp(F.lit(WINDOW_END),   "yyyy-MM-dd") + F.lit(86400)   # include the last day

def _in_window(df):
    df = df.withColumn("created_utc", F.col("created_utc").cast("long"))
    return df.filter(
        F.col("subreddit").isin(SUBREDDITS)              # safety net; design scales to all 4 subreddits
        & (F.col("created_utc") >= START) & (F.col("created_utc") < END)
    )

comments = (
    _in_window(spark.read.option("pathGlobFilter", "*_comments.zst").json(RAW_REDDIT))
    .select("created_utc", F.col("body").alias("text"))
)
submissions = (
    _in_window(spark.read.option("pathGlobFilter", "*_submissions.zst").json(RAW_REDDIT))
    .select("created_utc", F.concat_ws(" ", F.col("title"), F.col("selftext")).alias("text"))
)

reddit = (
    comments.unionByName(submissions)
    .filter(F.col("text").isNotNull() & ~F.trim("text").isin("", "[deleted]", "[removed]"))
    .withColumn("date",  F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("reddit"))
    .select("source", "month", "date", "text")
)
print("Reddit rows (filtered):", reddit.count())
reddit.show(3, truncate=80)

In [ ]:
# -------------------------------------------------------------
# Steam reviews (CSV). Verified schema:
#   ReviewID, SteamID, Rating, Review, "Date Posted" (M/d/yyyy), Hours Played, ...
# multiLine + escape='"' because reviews contain newlines and doubled quotes.
# (Rating = Recommended/Not Recommended kept aside for later VADER validation.)
# -------------------------------------------------------------
steam = (
    spark.read
    .option("header", True).option("multiLine", True).option("escape", '"')
    .csv(RAW_STEAM)
    .select(
        F.col("Review").alias("text"),
        F.to_date(F.col("Date Posted"), "M/d/yyyy").alias("date"),
    )
    .filter(F.col("text").isNotNull() & (F.length(F.trim("text")) > 0))
    .filter(F.col("date").between(WINDOW_START, WINDOW_END))   # drops out-of-window (e.g. 2024) + unparsable dates
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("steam"))
    .select("source", "month", "date", "text")
)
print("Steam rows (filtered):", steam.count())
steam.show(3, truncate=80)

In [ ]:
# -------------------------------------------------------------
# Combine both sources and persist (partitioned by source + month).
# Continue downstream with: spark.read.parquet(PROCESSED + "cleaned/")
# NEVER .collect()/.toPandas() on the full df — aggregate first.
# -------------------------------------------------------------
documents = reddit.unionByName(steam)

(
    documents.write.mode("overwrite")
    .partitionBy("source", "month")
    .parquet(PROCESSED + "cleaned/")
)
print("Written to:", PROCESSED + "cleaned/")

# Verify at aggregated level only
documents.groupBy("source").count().show()

### 1c · Patch Timeline (3rd Data Source)

Small, manually maintained table (`patch_name, release_date, platform, notes`) with the major
Cyberpunk 2077 patches in the window (1.0 Launch … 2.1). Stored in the repo under `data/sample/`.
Used later as patch markers on the sentiment time series and as anchor for the recovery metric.

In [ ]:
# -------------------------------------------------------------
# 4) Read Patch Timeline (3rd data source)
#    Small + manually maintained -> in the repo. Used later for
#    patch markers and the recovery metric (core thesis).
# -------------------------------------------------------------
patches = (
    spark.read.option("header", True).csv(PATCH_TIMELINE)
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("month", F.date_format("release_date", "yyyy-MM"))
    .filter(F.col("release_date").between(WINDOW_START, WINDOW_END))
)

# Small enough for toPandas (only ~9 rows) — used in Stage 5 for the overlay
patches.orderBy("release_date").show(truncate=False)

### Record Run Info
Which mode / which window the `results/` snapshot is based on (reproducibility).

In [ ]:
from datetime import datetime, timezone

Path(RESULTS, "RUN_INFO.txt").write_text(
    f"DATA_MODE = {DATA_MODE}\n"
    f"window    = {WINDOW_START} .. {WINDOW_END}\n"
    f"subreddits= {SUBREDDITS}\n"
    f"seed      = {SEED}\n"
    f"generated = {datetime.now(timezone.utc).isoformat()}\n"
)
print((Path(RESULTS) / "RUN_INFO.txt").read_text())

## 2 · Cleaning & Preprocessing  *(Phase 2 — TODO)*

- Normalize text (lowercasing, remove URLs/Markdown, whitespace)
- Language filter (English only — VADER is trained on English)
- Deduplication

## 3 · VADER Sentiment UDF  *(Phase 3 — TODO)*

VADER as Spark UDF on all comments/reviews; keep the `compound` score.

## 4 · Monthly/Source Aggregation  *(Phase 3 — TODO)*

Aggregate sentiment per `month` x `source` — **only here** does it become small enough for `.toPandas()`.

Writes: `results/sentiment_timeseries.csv` (`source, month, mean_sentiment, n`).

## 5 · Analysis & Visualization  *(Phase 3 — TODO)*

Each output -> exactly one file under `results/` (see report mapping in README):

- Two sentiment time series + patch markers  -> `results/figures/`, `results/patch_overlay.csv`
- Review-bombing anomaly flag (volume anomaly, `ANOMALY_Z_THRESHOLD`)  -> `results/anomaly_flags.csv`
- Sentiment without flagged windows  -> `results/sentiment_excl_bombing.csv`
- Recovery metric (core thesis)  -> `results/recovery_metric.csv`
- Topic analysis (TF-IDF / LDA with `setSeed(SEED)`, `LDA_NUM_TOPICS`)  -> `results/topics.csv`